# Optional HMM Regime Detection

This notebook estimates hidden residual regimes from the dynamic residual series and compares strategy behavior across inferred regimes. The regime labels are statistical descriptions of the residual process, not verified market states.

The main use case is a conservative filter: only allow trades when the estimated mean-reverting regime probability is sufficiently high.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.hmm import (
    HMMConfig,
    fit_hmm_by_triplet,
    apply_regime_trade_filter,
    summarize_strategy_performance_by_regime,
    plot_regime_timeline,
    plot_performance_by_regime,
)
from src.database import (
    connect_database,
    store_hmm_regime_probabilities,
    store_hmm_regime_parameters,
    store_hmm_strategy_performance,
)

## Data inputs

The preferred input is a residual z-score series by date and triplet. If the project has Kalman residuals but no explicit z-score column, the notebook builds a rolling residual z-score using past and current observations only.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "figures"
DB_PATH = PROJECT_ROOT / "data" / "triangular_stat_arb.sqlite"

residual_path = DATA_DIR / "hmm_placeholder_residual_z_scores.csv"
kalman_path = DATA_DIR / "kalman_residual_table.csv"
trade_log_path = DATA_DIR / "ml_backtest_trade_log.csv"

if residual_path.exists():
    residuals = pd.read_csv(residual_path, parse_dates=["date"])
elif kalman_path.exists():
    residuals = pd.read_csv(kalman_path, parse_dates=["date"])
    residuals = residuals.sort_values(["triplet_id", "date"]).copy()
    rolling_mean = residuals.groupby("triplet_id")["residual"].transform(lambda s: s.rolling(60, min_periods=20).mean())
    rolling_std = residuals.groupby("triplet_id")["residual"].transform(lambda s: s.rolling(60, min_periods=20).std())
    residuals["residual_z_score"] = (residuals["residual"] - rolling_mean) / rolling_std.replace(0.0, np.nan)
    residuals = residuals.dropna(subset=["residual_z_score"])
else:
    raise FileNotFoundError("Expected residual z-score inputs or Kalman residual outputs.")

residuals.head()

## Fit Gaussian HMM by triplet

The model uses three hidden states. The highest-variance state is labeled as a volatile breakdown regime. Among the remaining states, the state with mean closest to zero is labeled mean-reverting, and the other state is labeled trending.

In [ ]:
config = HMMConfig(
    n_states=3,
    max_iter=50,
    tolerance=1e-5,
    transition_stickiness=0.90,
    feature_column="residual_z_score",
)

hmm_output = fit_hmm_by_triplet(
    residuals,
    value_column="residual_z_score",
    date_column="date",
    triplet_column="triplet_id",
    config=config,
)

regime_probabilities = hmm_output["regime_probabilities"]
regime_parameters = hmm_output["regime_parameters"]

regime_probabilities.to_csv(DATA_DIR / "hmm_regime_probability_table.csv", index=False)
regime_parameters.to_csv(DATA_DIR / "hmm_regime_parameters.csv", index=False)

regime_parameters.head()

## Regime timeline

The chart shows posterior regime probabilities through time for one triplet. This is useful for checking whether the filter is smooth enough to be plausible and reactive enough to identify breakdown periods.

In [ ]:
if not regime_probabilities.empty:
    selected_triplet = str(regime_probabilities["triplet_id"].iloc[0])
    plot_regime_timeline(
        regime_probabilities,
        FIGURE_DIR / "regime_timeline_chart.png",
        triplet_id=selected_triplet,
    )

FIGURE_DIR / "regime_timeline_chart.png"

## Strategy performance by inferred regime

The regime filter is evaluated on the existing ML backtest trade log. This is not a replacement for a full walk-forward backtest. It is a diagnostic: it asks whether trades occurring during high mean-reverting probability regimes behave differently from trades occurring outside those regimes.

In [ ]:
if trade_log_path.exists() and not regime_probabilities.empty:
    trade_log = pd.read_csv(trade_log_path, parse_dates=["event_date"])
    filtered_trades = apply_regime_trade_filter(trade_log, regime_probabilities, threshold=0.60)
    performance_by_regime = summarize_strategy_performance_by_regime(trade_log, regime_probabilities, threshold=0.60)

    filtered_trades.to_csv(DATA_DIR / "hmm_regime_filtered_trades.csv", index=False)
    performance_by_regime.to_csv(DATA_DIR / "hmm_strategy_performance_by_regime.csv", index=False)

    plot_performance_by_regime(performance_by_regime, FIGURE_DIR / "performance_by_regime.png")
else:
    filtered_trades = pd.DataFrame()
    performance_by_regime = pd.DataFrame()

performance_by_regime

## SQL storage

The regime probabilities, fitted state parameters, and strategy-by-regime summary can be stored for later analysis.

In [ ]:
try:
    with connect_database(DB_PATH) as conn:
        store_hmm_regime_probabilities(conn, regime_probabilities, if_exists="replace")
        store_hmm_regime_parameters(conn, regime_parameters, if_exists="replace")
        if not performance_by_regime.empty:
            store_hmm_strategy_performance(conn, performance_by_regime, if_exists="replace")
        conn.commit()
except Exception as exc:
    print(f"SQL storage skipped: {exc}")

## Research interpretation

The HMM should be treated as a diagnostic regime layer, not as proof that the market has exactly three true states. The key test is whether high mean-reverting regime probability aligns with better residual trade behavior out of sample. If the event set is small or the regime assignments are unstable, the filter should remain optional.